# RAG Document Assistant — Exploration Notebook

This notebook walks through the RAG pipeline end-to-end, outside of the FastAPI app, so you can inspect each stage:

1. Setup & prerequisite checks (Ollama reachable, models pulled)
2. Load and chunk a sample document
3. Generate embeddings with Ollama and inspect them
4. Store and query chunks in ChromaDB
5. Run a full RAG query (retrieve + generate) and inspect grounding

> Run this from the project root (`rag-document-assistant/`) so the `backend` and `config` imports resolve correctly.

## 1. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import ollama
from config import settings

print("Ollama base URL:", settings.OLLAMA_BASE_URL)
print("Chat model:", settings.OLLAMA_MODEL)
print("Embedding model:", settings.OLLAMA_EMBED_MODEL)

In [ ]:
# Check Ollama is running and required models are available
client = ollama.Client(host=settings.OLLAMA_BASE_URL)

try:
    models_info = client.list()
    available = [m["model"] for m in models_info.get("models", [])]
    print("Available Ollama models:", available)

    if not any(settings.OLLAMA_MODEL in m for m in available):
        print(f"\n⚠️  '{settings.OLLAMA_MODEL}' not found. Run: ollama pull {settings.OLLAMA_MODEL}")
    if not any(settings.OLLAMA_EMBED_MODEL in m for m in available):
        print(f"⚠️  '{settings.OLLAMA_EMBED_MODEL}' not found. Run: ollama pull {settings.OLLAMA_EMBED_MODEL}")
except Exception as e:
    print("❌ Could not reach Ollama. Is it running? Try `ollama serve`.")
    print(e)

## 2. Load and chunk a sample document

In [ ]:
sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval
with text generation. Instead of relying purely on what a language model memorized during
training, RAG systems first retrieve relevant chunks of text from an external knowledge
base, then feed those chunks to the model as context alongside the user's question.

This approach has two major benefits. First, it reduces hallucination, because the model
is grounded in real, retrieved evidence rather than generating facts from memory. Second,
it allows the system's knowledge to be updated simply by adding new documents to the
retrieval index, without retraining the underlying model at all.

A typical RAG pipeline has three stages: ingestion (splitting documents into chunks and
embedding them into vectors), retrieval (embedding the user's query and finding the most
similar chunks via vector search), and generation (prompting an LLM with the retrieved
chunks as context to produce a grounded answer).
"""

from backend.document_processor import chunk_text

chunks = chunk_text(sample_text, chunk_size=40, chunk_overlap=8)
print(f"Created {len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    print(f"--- Chunk {i} ---\n{c}\n")

## 3. Generate embeddings and inspect them

In [ ]:
embedding_response = client.embeddings(model=settings.OLLAMA_EMBED_MODEL, prompt=chunks[0])
vector = embedding_response["embedding"]

print("Embedding dimensionality:", len(vector))
print("First 10 values:", vector[:10])

## 4. Store and query chunks in ChromaDB

In [ ]:
from backend.vector_store import VectorStore

vs = VectorStore(
    persist_dir="./demo_chroma_db",
    collection_name="demo_collection",
    embed_model=settings.OLLAMA_EMBED_MODEL,
    ollama_base_url=settings.OLLAMA_BASE_URL,
)

vs.add_chunks(document_id="demo-doc-1", filename="rag_intro.txt", file_type=".txt", chunks=chunks)
print("Total chunks stored:", vs.total_chunks())

In [ ]:
query = "Why does RAG reduce hallucination?"
results = vs.query(query_text=query, top_k=2)

for r in results:
    print(f"Similarity: {r['similarity_score']} | chunk #{r['chunk_index']}")
    print(r["text"])
    print("---")

## 5. Full RAG query: retrieve + generate

In [ ]:
from backend.llm_service import LLMService
from backend.rag_engine import RAGEngine

llm = LLMService(model=settings.OLLAMA_MODEL, base_url=settings.OLLAMA_BASE_URL)
engine = RAGEngine(
    vector_store=vs,
    llm_service=llm,
    chunk_size=settings.CHUNK_SIZE,
    chunk_overlap=settings.CHUNK_OVERLAP,
    top_k=settings.TOP_K_RESULTS,
)

response = engine.query("What are the three stages of a RAG pipeline?")

print("ANSWER:\n", response["answer"])
print("\nSOURCES USED:")
for s in response["sources"]:
    print(f" - {s['filename']} chunk #{s['chunk_index']} (score={s['similarity_score']})")

## 6. Cleanup (optional)

Removes the demo collection created in this notebook so it doesn't linger on disk.

In [ ]:
import shutil
vs.reset()
shutil.rmtree("./demo_chroma_db", ignore_errors=True)
print("Demo collection removed.")